# Yelp Pipeline - 04: Recommender Systems (Content-Based, Collaborative SVD, Deep Learning Keras)

Questo notebook implementa tre paradigmi complementari di raccomandazione per le attività gastronomiche:
1. **Feature Engineering degli Attributi Nidificati**: Estrazione e decodifica dei dizionari attributi (`BusinessParking`, `Ambience`, `GoodForMeal`, `Dietary`, `Music`, `Alcohol`, `PriceRange`).
2. **Content-Based Filtering**: Calcolo della similarità tra attività su spazio vettoriale misto (attributi + categorie) tramite **K-Nearest Neighbors (KNN)** con metrica del coseno.
3. **Collaborative Filtering Matriciale (TruncatedSVD)**: Decomposizione della matrice di utilità Utenti-Ristoranti nello spazio latente e calcolo della matrice di correlazione tra item.
4. **Collaborative Filtering con Deep Learning (Keras / TensorFlow)**: Modello neurale ad embedding latenti per utenti e ristoranti con strati densi per interazioni non lineari.
5. **Esportazione Modelli e Metriche**: Serializzazione su Google Drive per inferenza e benchmark comparativo.

## Configurazione Ambiente e Dipendenze

In [1]:
%pip install -q scikit-learn tensorflow joblib loguru pandas numpy tabulate

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.6/61.6 kB 6.5 MB/s eta 0:00:00


## Montaggio Google Drive e Configurazione

In [2]:
import os
import sys
import json
import time
import ast
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from loguru import logger
import joblib
import polars as pl

from sklearn.neighbors import NearestNeighbors
from sklearn.decomposition import TruncatedSVD
from sklearn.metrics import mean_squared_error, mean_absolute_error
from sklearn.model_selection import train_test_split

import tensorflow as tf
from tensorflow.keras import layers, models, optimizers

os.environ['TZ'] = 'Europe/Rome'
time.tzset()

logger.remove()
logger.add(
    sys.stdout,
    colorize=True,
    format="<green>{time:YYYY-MM-DD HH:mm:ss}</green> | <level>{level: <8}</level> | <cyan>{message}</cyan>"
)

RANDOM_SEED = 42
np.random.seed(RANDOM_SEED)
tf.random.set_seed(RANDOM_SEED)

try:
    from google.colab import drive
    if not Path('/content/drive/MyDrive').exists():
        drive.mount('/content/drive')
    else:
        print("Google Drive già connesso e accessibile.")
except Exception as e:
    logger.warning(f"Avviso montaggio Drive: {e}")

candidates = [
    Path('/content/drive/MyDrive/yelp_data'),
    Path('/content/drive/MyDrive/data'),
    Path('/content/drive/MyDrive'),
    Path('./yelp_data'),
    Path('./data'),
    Path('.')
]

DRIVE_DIR = None
for p in candidates:
    if (p / 'yelp_businesses.parquet').exists() and (p / 'yelp_reviews.parquet').exists():
        DRIVE_DIR = p
        break

if DRIVE_DIR is None:
    DRIVE_DIR = Path('/content/drive/MyDrive/yelp_data') if Path('/content/drive/MyDrive').exists() else Path('./yelp_data')

DRIVE_DIR.mkdir(parents=True, exist_ok=True)
logger.info(f"Directory artifact selezionata: {DRIVE_DIR}")

Mounted at /content/drive
2026-10-05 09:22:34 | INFO     | Directory artifact selezionata: /content/drive/MyDrive/yelp_data


## Caricamento Dati

In [3]:
required_parquets = ['yelp_businesses.parquet', 'yelp_reviews.parquet']
missing = [f for f in required_parquets if not (DRIVE_DIR / f).exists()]
if missing:
    raise FileNotFoundError(
        f"Artifact mancanti su Drive: {missing}. Esegui prima Yelp_01_Ingestion_Preprocessing.ipynb!"
    )

t0 = time.perf_counter()
df_businesses_pl = pl.read_parquet(DRIVE_DIR / 'yelp_businesses.parquet')
df_reviews = pl.read_parquet(
    DRIVE_DIR / 'yelp_reviews.parquet',
    columns=['user_id', 'business_id', 'stars', 'date']
)

df_businesses = df_businesses_pl.to_pandas()

logger.success(f"Dati caricati in {time.perf_counter() - t0:.2f}s | Ristoranti: {len(df_businesses):,d} | Recensioni: {df_reviews.height:,d}")

2026-10-05 09:23:21 | SUCCESS  | Dati caricati in 47.87s | Ristoranti: 46,146 | Recensioni: 4,185,052


## 1. Feature Engineering: Estrazione Attributi Nidificati

In [4]:
def extract_nested_dict(attr_val, key):
    if not isinstance(attr_val, dict):
        if isinstance(attr_val, str) and (attr_val.startswith('{') or attr_val.startswith("u'{")):
            try:
                attr_val = ast.literal_eval(attr_val)
            except Exception:
                return {}
        else:
            return {}
    val = attr_val.get(key, {})
    if isinstance(val, str) and val.startswith('{'):
        try:
            return ast.literal_eval(val)
        except Exception:
            return {}
    return val if isinstance(val, dict) else {}

t0 = time.perf_counter()
# reset dell'indice per garantire allineamento perfetto 1:1 nel concat
b_df = df_businesses[['business_id', 'name', 'city', 'stars', 'review_count', 'categories', 'attributes']].copy().reset_index(drop=True)

#categorie culinarie dummy (top 35)
cats_dummies = b_df['categories'].fillna('').str.get_dummies(sep=',').reset_index(drop=True)
top_cats = cats_dummies.sum().sort_values(ascending=False).head(35).index
df_top_cats = cats_dummies[top_cats].copy()
df_top_cats.columns = [f"cat_{c.strip().lower()}" for c in df_top_cats.columns]

#parsing dizionari nidificati
nested_attrs = ['BusinessParking', 'Ambience', 'GoodForMeal', 'Dietary', 'Music']
nested_dfs = []

for attr_name in nested_attrs:
    extracted = b_df['attributes'].apply(lambda x: extract_nested_dict(x, attr_name))
    expanded = pd.json_normalize(extracted).fillna(False)
    expanded = expanded.map(lambda v: 1 if v in [True, 'True', 'true', 1] else 0)
    expanded.columns = [f"{attr_name.lower()}_{c}" for c in expanded.columns]
    nested_dfs.append(expanded)

df_nested_all = pd.concat(nested_dfs, axis=1).reset_index(drop=True)

#attributi standard
def get_simple_attr(attr_val, key, default='None'):
    if isinstance(attr_val, str) and attr_val.startswith('{'):
        try:
            attr_val = ast.literal_eval(attr_val)
        except Exception:
            return default
    if isinstance(attr_val, dict):
        return str(attr_val.get(key, default))
    return default

b_df['price_range'] = b_df['attributes'].apply(lambda x: get_simple_attr(x, 'RestaurantsPriceRange2', '2'))
b_df['alcohol'] = b_df['attributes'].apply(lambda x: get_simple_attr(x, 'Alcohol', 'none'))
b_df['takeout'] = b_df['attributes'].apply(lambda x: get_simple_attr(x, 'RestaurantsTakeOut', 'False'))

simple_dummies = pd.get_dummies(b_df[['price_range', 'alcohol', 'takeout']], drop_first=True).reset_index(drop=True)

content_df = pd.concat([
    b_df[['business_id', 'name', 'city', 'stars']],
    df_top_cats,
    df_nested_all,
    simple_dummies
], axis=1).fillna(0)

logger.success(f"Feature matrix costruita in {time.perf_counter() - t0:.2f}s | Dimensioni: {content_df.shape}")

/tmp/ipykernel_788/553378326.py:34: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  expanded = pd.json_normalize(extracted).fillna(False)


2026-10-05 09:23:49 | SUCCESS  | Feature matrix costruita in 28.02s | Dimensioni: (46146, 79)


## 2. Content-Based Filtering (KNN + Cosine)

In [5]:
feat_cols = [c for c in content_df.columns if c not in ['business_id', 'name', 'city']]
X_features = content_df[feat_cols].values

t0 = time.perf_counter()
knn = NearestNeighbors(n_neighbors=10, metric='cosine', algorithm='brute')
knn.fit(X_features)
t_knn_train = time.perf_counter() - t0

def recommend_content_based(restaurant_name, top_k=5):
    t_start = time.perf_counter()
    matches = content_df[content_df['name'].str.lower().str.contains(restaurant_name.lower(), na=False)]
    if len(matches) == 0:
        return pd.DataFrame(), 0.0

    idx = matches.index[0]
    vec = X_features[idx].reshape(1, -1)
    dists, indices = knn.kneighbors(vec, n_neighbors=top_k + 1)

    rec_indices = indices[0][1:top_k + 1]
    rec_dists = dists[0][1:top_k + 1]

    results = content_df.iloc[rec_indices][['business_id', 'name', 'city', 'stars']].copy()
    results['similarity'] = (1.0 - rec_dists).round(4)
    return results, time.perf_counter() - t_start

test_name = content_df['name'].iloc[0]
recs_cb, t_knn_inf = recommend_content_based(test_name, top_k=5)
logger.info(f"Raccomandazioni Content-Based per '{test_name}' ({t_knn_inf*1000:.1f} ms):")
display(recs_cb)

2026-10-05 09:23:50 | INFO     | Raccomandazioni Content-Based per 'St Honore Pastries' (50.0 ms):


,business_id,name,city,stars,similarity
4656,7hpUoYdAVToZXwuPRgoqdA,Chapterhouse Café & Gallery,Philadelphia,4.5,0.9783
25058,3yW8-dUi1wwCEf0URxUA7A,Kung Fu Tea,Philadelphia,3.5,0.9778
17891,aA-6urKmmlZw9STPOHZNCg,up-RYES Bagel & Deli,Bryn Mawr,4.0,0.9623
44529,KfnMkNJrQQMmqNs34aAXBw,Nourishing Storm Studio & Cafe,Hatboro,4.5,0.9615
39036,3NsTKp85H2FUjr1SURf0NQ,Greenstreet Coffee Co.,Philadelphia,4.5,0.9615


## 3. Collaborative Filtering Matriciale (TruncatedSVD)

In [6]:
# selezione del nucleo denso di recensioni per Matrix Factorization in Polars
# calcolo ultra veloce dei conteggi
top_users = df_reviews.group_by('user_id').len().sort('len', descending=True).head(5000)['user_id'].to_list()
top_businesses = df_reviews.group_by('business_id').len().sort('len', descending=True).head(2500)['business_id'].to_list()

df_dense = df_reviews.filter(
    pl.col('user_id').is_in(top_users) & pl.col('business_id').is_in(top_businesses)
).to_pandas()

logger.info(f"Sottogruppo denso per Collaborative Filtering: {len(df_dense):,d} valutazioni ({len(top_users):,d} utenti x {len(top_businesses):,d} ristoranti)")

#costruzione della matrice di utilità (dimensione controllata 2,500 x 5,000, ~100 MB RAM)
utility = df_dense.pivot_table(index='business_id', columns='user_id', values='stars', fill_value=0)

t0 = time.perf_counter()
RANDOM_SEED = 42 # Fallback
N_COMPONENTS = min(30, utility.shape[0] - 1, utility.shape[1] - 1)
svd = TruncatedSVD(n_components=N_COMPONENTS, random_state=RANDOM_SEED)
latent = svd.fit_transform(utility)
corr_matrix = np.corrcoef(latent)
t_svd_train = time.perf_counter() - t0

def recommend_svd(restaurant_name, top_k=5):
    t_start = time.perf_counter()
    matches = df_businesses[df_businesses['name'].str.lower().str.contains(restaurant_name.lower(), na=False)]
    if len(matches) == 0:
        return pd.DataFrame(), 0.0

    avail = [b for b in matches['business_id'] if b in utility.index]
    if not avail:
        return pd.DataFrame(), 0.0
    bid = avail[0]

    bid_idx = list(utility.index).index(bid)
    corrs = corr_matrix[bid_idx]
    top_idx = corrs.argsort()[::-1][1:top_k + 1]

    rec_bids = [utility.index[i] for i in top_idx]
    results = df_businesses[df_businesses['business_id'].isin(rec_bids)][['business_id', 'name', 'city', 'stars']].copy()
    results['correlation'] = [corrs[i].round(4) for i in top_idx]
    return results, time.perf_counter() - t_start

test_name = df_businesses[df_businesses['business_id'].isin(utility.index)]['name'].iloc[0]
recs_svd, t_svd_inf = recommend_svd(test_name, top_k=5)
logger.info(f"SVD fit in {t_svd_train:.2f}s | Varianza spiegata: {svd.explained_variance_ratio_.sum()*100:.1f}%")
if not recs_svd.empty:
    display(recs_svd)

2026-10-05 09:23:51 | INFO     | Sottogruppo denso per Collaborative Filtering: 182,276 valutazioni (5,000 utenti x 2,500 ristoranti)
2026-10-05 09:23:52 | INFO     | SVD fit in 1.13s | Varianza spiegata: 22.5%


,business_id,name,city,stars,correlation
2441,W8Z4rXYkmZlEVSaxIVjyvg,Redwood Rotisserie + Grill,Reno,4.0,0.9999
5273,29fX9PdevQa5LZavVDJsbw,Roberto's Taco Shop,Reno,3.0,0.9997
13491,EYs_T4NU2BbaUF7vUDI55Q,JJ Sushi,Sparks,4.0,0.9997
20238,5drbv3fz5FTvp_Z3d3aPSQ,Tacos Jalisco Grill & Cantina,Reno,4.0,0.9996
37704,Pr-OrQ0-hNPFdp8bVZN6Lg,Zozo's Ristorante,Reno,4.0,0.9995


## 4. Collaborative Filtering con Deep Learning (Keras Embeddings)

In [7]:
user_list = df_dense['user_id'].unique().tolist()
item_list = df_dense['business_id'].unique().tolist()

u2idx = {u: i for i, u in enumerate(user_list)}
b2idx = {b: i for i, b in enumerate(item_list)}

df_dense['u_idx'] = df_dense['user_id'].map(u2idx)
df_dense['b_idx'] = df_dense['business_id'].map(b2idx)

num_u = len(u2idx)
num_b = len(b2idx)

train_data, test_data = train_test_split(df_dense, test_size=0.20, random_state=RANDOM_SEED)

X_tr = [train_data['u_idx'].values, train_data['b_idx'].values]
y_tr = train_data['stars'].values.astype('float32')

X_te = [test_data['u_idx'].values, test_data['b_idx'].values]
y_te = test_data['stars'].values.astype('float32')

#rete neurale Keras con Embedding Latenti
EMB_DIM = 32

u_in = layers.Input(shape=(1,), name='user_input')
u_emb = layers.Embedding(num_u, EMB_DIM, name='user_emb')(u_in)
u_bias = layers.Embedding(num_u, 1, name='user_bias')(u_in)

b_in = layers.Input(shape=(1,), name='item_input')
b_emb = layers.Embedding(num_b, EMB_DIM, name='item_emb')(b_in)
b_bias = layers.Embedding(num_b, 1, name='item_bias')(b_in)

dot_prod = layers.Dot(axes=2)([u_emb, b_emb])
add_b = layers.Add()([dot_prod, u_bias, b_bias])
flat = layers.Flatten()(add_b)

concat_emb = layers.Concatenate()([layers.Flatten()(u_emb), layers.Flatten()(b_emb)])
dense_h = layers.Dense(32, activation='relu')(concat_emb)
dense_h = layers.Dropout(0.2)(dense_h)
dense_out = layers.Dense(1)(dense_h)

pred_out = layers.Activation('linear')(layers.Add()([flat, dense_out]))

dl_recommender = models.Model(inputs=[u_in, b_in], outputs=pred_out)
dl_recommender.compile(optimizer=optimizers.Adam(learning_rate=0.005), loss='mean_squared_error')

t0 = time.perf_counter()
es = tf.keras.callbacks.EarlyStopping(monitor='val_loss', patience=3, restore_best_weights=True)
dl_recommender.fit(
    X_tr, y_tr,
    epochs=8,
    batch_size=256,
    validation_split=0.15,
    callbacks=[es],
    verbose=0
)
t_dl_train = time.perf_counter() - t0

y_pred_dl = dl_recommender.predict(X_te, verbose=0).flatten()
rmse_score = float(np.sqrt(mean_squared_error(y_te, y_pred_dl)))
mae_score = float(mean_absolute_error(y_te, y_pred_dl))

logger.success(f"Keras Recommender addestrato in {t_dl_train:.2f}s | RMSE: {rmse_score:.4f} | MAE: {mae_score:.4f}")

2026-10-05 09:24:08 | SUCCESS  | Keras Recommender addestrato in 10.09s | RMSE: 0.9177 | MAE: 0.7336


## 5. Inferenza Personalizzata Utente

In [8]:
def recommend_for_user_dl(user_id, top_k=5):
    t_start = time.perf_counter()
    if user_id not in u2idx:
        return pd.DataFrame(), 0.0

    u_idx = u2idx[user_id]
    all_items = np.arange(num_b)
    user_arr = np.full(num_b, u_idx)

    ratings_pred = dl_recommender.predict([user_arr, all_items], verbose=0).flatten()
    best_indices = ratings_pred.argsort()[::-1][:top_k]

    rec_bids = [item_list[i] for i in best_indices]
    results = df_businesses[df_businesses['business_id'].isin(rec_bids)][['business_id', 'name', 'city', 'stars']].copy()
    results['predicted_stars'] = ratings_pred[best_indices].round(2)
    return results, time.perf_counter() - t_start

test_user = user_list[0]
recs_user, t_dl_inf = recommend_for_user_dl(test_user, top_k=5)
logger.info(f"Top 5 raccomandazioni per utente {test_user} ({t_dl_inf*1000:.1f} ms):")
display(recs_user)

2026-10-05 09:24:08 | INFO     | Top 5 raccomandazioni per utente ZGjgfSvjQK886kiTzLwfLQ (396.1 ms):


,business_id,name,city,stars,predicted_stars
4791,IhrTLL6az3q192ik4P05IQ,The Chef and I,Nashville,4.5,5.08
17337,Esf3-D_44pArPd9GqysoCg,Duchess Bake Shop,Edmonton,4.5,4.96
21788,tBQcVd9c0EHKAXLZn_v54g,Bibou,Philadelphia,4.5,4.96
27424,aFhPwy7OhyKah4cJY2QdEQ,Stonehouse Restaurant,Santa Barbara,4.5,4.94
29340,xZileQFJ7h-0QKjtGc9xgg,Arnaud's French 75 Bar,New Orleans,4.5,4.92


## 6. Salvataggio Modelli e Metriche

In [9]:
joblib.dump(knn, DRIVE_DIR / 'content_knn_model.joblib')
joblib.dump(svd, DRIVE_DIR / 'svd_model.joblib')
dl_recommender.save(DRIVE_DIR / 'keras_recommender.keras')

metrics = {
    'content_knn': {
        'train_time_s': float(t_knn_train),
        'infer_time_ms': float(t_knn_inf * 1000),
        'metric': 'Cosine Similarity',
        'n_features': int(X_features.shape[1])
    },
    'collaborative_svd': {
        'train_time_s': float(t_svd_train),
        'infer_time_ms': float(t_svd_inf * 1000),
        'explained_variance': float(svd.explained_variance_ratio_.sum()),
        'components': int(N_COMPONENTS)
    },
    'collaborative_dl_keras': {
        'train_time_s': float(t_dl_train),
        'infer_time_ms': float(t_dl_inf * 1000),
        'rmse': float(rmse_score),
        'mae': float(mae_score),
        'embedding_dim': int(EMB_DIM)
    }
}

with open(DRIVE_DIR / 'recommender_metrics.json', 'w', encoding='utf-8') as f:
    json.dump(metrics, f, indent=2)

logger.success("Modelli e metriche dei Recommender Systems salvati con successo su Google Drive.")

2026-10-05 09:24:12 | SUCCESS  | Modelli e metriche dei Recommender Systems salvati con successo su Google Drive.
